# <h1 style="color: green">Neural Network from Scratch</h1>
### Single Hidden Layer | Regression | TensorFlow

Implementing a single hidden layer neural network using TensorFlow's `GradientTape` for manual backpropagation, trained on the **California Housing** dataset.

---

## Results

| Model | RMSE |
|-------|------|
| **My Implementation** | 0.736 |
| TensorFlow (Keras) | 0.739 |
| Linear Regression | 0.746 |

> My implementation matches native TensorFlow within 0.003 RMSE.

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
import numpy as np
import tensorflow as tf

In [4]:
print(tf.config.list_physical_devices())

[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]


E0000 00:00:1779525214.588481    1135 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [5]:
from tqdm import tqdm

In [6]:
import matplotlib.pyplot as plt

## My Neural Network 

In [7]:
class FitError(Exception):
    pass

class NN:
    """
    Single hidden layer neural network for regression.
    
    """
    def __init__(self, hidden_shape, activation = 'relu', random_state = None):
        """
    Initialize the Neural Network.
    
        Parameters
        ----------
        hidden_shape : int
            Number of neurons in the hidden layer.
        activation : str, default='relu'
            Activation function for hidden layer. One of ['relu', 'tanh', 'sigmoid'].
        random_state : int or None, default=None
            Seed for reproducibility. If None, results may vary between runs.
    """
        self.hidden_shape = hidden_shape
        self.random_state = random_state
        if activation == 'tanh':
            self.activation = tf.keras.activations.tanh
        elif activation == 'relu':
            self.activation = tf.keras.activations.relu
        elif activation == 'sigmoid':
            self.activation = tf.keras.activations.sigmoid
        self._w11 = None
        self._w12 = None
        self._w01 = None
        self._w02 = None 
        self.__is_fitted = False
        self._cost = []
        
    def fit(self, x, y, lr = 0.03, it = 50, batch_size = 32, show_execution = False):
        """
            Train the neural network.
        
            Parameters
            ----------
            x : np.ndarray of shape (n_samples, n_features)
                Training features.
            y : np.ndarray of shape (n_samples, 1)
                Target values.
            lr : float, default=0.03
                Learning rate.
            it : int, default=100
                Number of epochs.
            batch_size : int, default=32
                Number of samples per gradient update.
        """
        
        self._cost = []
        self.__is_fitted = True
        y = y.reshape(-1, 1) if y.ndim == 1 else y
        if self.random_state: 
            tf.random.set_seed(self.random_state)
        #entadrenq unenq mxn dataset, 10 neurons, => mxn @  ? =  m x neurons  (n x neurons)
        #                                           m x neurons @ ?  = m x 1 (neurons x 1)
        
        
        
        w11 = tf.Variable(tf.random.normal((x.shape[1], self.hidden_shape), dtype = 'float32'))  
        w12 = tf.Variable(tf.random.normal((self.hidden_shape, y.shape[1]), dtype = 'float32'))        

        w01 = tf.Variable(tf.random.normal((1, self.hidden_shape), dtype = 'float32'))
        w02 = tf.Variable(tf.random.normal((1, y.shape[1]), dtype = 'float32'))       

        params = [w11, w12, w01, w02]
        
        for i in range(it):
            a = range(x.shape[0] // batch_size + (x.shape[0] % batch_size != 0))
            t = tqdm(a) if show_execution else a
            for j in t:
                with tf.GradientTape() as tape:
                    h = self.activation(x[batch_size*j: batch_size*(j+1)] @ w11 + w01) 
                    ypr = h @ w12 + w02
                    mse = tf.reduce_mean(tf.square(y[batch_size*j: batch_size*(j+1)] - ypr))
                    self._cost.append(mse)
                dparams = tape.gradient(mse, params)
                
                for p, d in zip(params, dparams):
                    p.assign_sub(lr*d)
                if show_execution: 
                    t.set_description(f"{i}/{it} | MSE={mse.numpy():.8f}")


        self._w11 = w11
        self._w12 = w12
        self._w01 = w01
        self._w02 = w02

    def predict(self, x):
        if self.__is_fitted is True:
            h = self.activation(x @ self._w11 + self._w01)
            return h @ self._w12 + self._w02
        else:
            raise FitError('You have to fit the model before prediction')
    def display_cost(self):
        if self.__is_fitted is True:
            plt.plot(self._cost, color = 'red')
            plt.title(f'MSE Loss function with {self.activation} activation function')
            plt.xlabel('Iterations')
            plt.ylabel('Cost')
            plt.show()
        else:
            raise FitError('You have to fit the model before Displaying Cost')

## California Housing Dataset

In [8]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import root_mean_squared_error

In [9]:
data = fetch_california_housing()
x = data.data
y = data.target
features = data.feature_names
target_name = data.target_names
print(features)
print(target_name)

['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']
['MedHouseVal']


In [10]:
x.shape, y.shape

((20640, 8), (20640,))

In [11]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 0)

## <u>Trying out my model</u>
### 1. Without scaling
### 2. With scaling

In [28]:
n1 = NN(hidden_shape = 10, activation='tanh', random_state=0)
n1.fit(x_train, y_train, lr = 0.001, show_execution = True)
ypr1 = n1.predict(x_test)
print(f'RMSE: {root_mean_squared_error(y_test, ypr1)}')

0/50 | MSE=1.15815592: 100%|█████████████████████████████████████████████████████████| 516/516 [00:02<00:00, 193.47it/s]
1/50 | MSE=1.15785253: 100%|█████████████████████████████████████████████████████████| 516/516 [00:02<00:00, 206.30it/s]
2/50 | MSE=1.15764260: 100%|█████████████████████████████████████████████████████████| 516/516 [00:02<00:00, 188.40it/s]
3/50 | MSE=1.15745842: 100%|█████████████████████████████████████████████████████████| 516/516 [00:02<00:00, 190.80it/s]
4/50 | MSE=1.15713573: 100%|█████████████████████████████████████████████████████████| 516/516 [00:02<00:00, 185.28it/s]
5/50 | MSE=1.15711284: 100%|█████████████████████████████████████████████████████████| 516/516 [00:02<00:00, 187.06it/s]
6/50 | MSE=1.15709448: 100%|█████████████████████████████████████████████████████████| 516/516 [00:02<00:00, 186.97it/s]
7/50 | MSE=1.15707767: 100%|█████████████████████████████████████████████████████████| 516/516 [00:02<00:00, 190.24it/s]
8/50 | MSE=1.15706277: 100%|████

RMSE: 1.1468750103064465


In [23]:
sc = MinMaxScaler()
x_train, x_test, y_train_, y_test_ = train_test_split(x, y, test_size = 0.2, random_state = 42)
x_train_sc = sc.fit_transform(x_train)
x_test_sc = sc.transform(x_test)

sc_y = StandardScaler()
y_train_sc = sc_y.fit_transform(y_train_.reshape(-1,1)).ravel()


In [32]:
n2 = NN(hidden_shape = 10, random_state=0, activation = 'tanh')
n2.fit(x_train_sc, y_train_sc, lr = 0.001, show_execution = True)
ypr2 = sc_y.inverse_transform(n2.predict(x_test_sc).numpy())

print(f'RMSE: {root_mean_squared_error(y_test_, ypr2)}')

0/50 | MSE=0.69519341: 100%|█████████████████████████████████████████████████████████| 516/516 [00:02<00:00, 184.00it/s]
1/50 | MSE=0.56510752: 100%|█████████████████████████████████████████████████████████| 516/516 [00:02<00:00, 194.81it/s]
2/50 | MSE=0.50443625: 100%|█████████████████████████████████████████████████████████| 516/516 [00:02<00:00, 188.89it/s]
3/50 | MSE=0.47038031: 100%|█████████████████████████████████████████████████████████| 516/516 [00:03<00:00, 143.16it/s]
4/50 | MSE=0.45038179: 100%|█████████████████████████████████████████████████████████| 516/516 [00:02<00:00, 185.34it/s]
5/50 | MSE=0.43886966: 100%|█████████████████████████████████████████████████████████| 516/516 [00:02<00:00, 174.13it/s]
6/50 | MSE=0.43257892: 100%|█████████████████████████████████████████████████████████| 516/516 [00:02<00:00, 174.23it/s]
7/50 | MSE=0.42943349: 100%|█████████████████████████████████████████████████████████| 516/516 [00:02<00:00, 197.14it/s]
8/50 | MSE=0.42811620: 100%|████

RMSE: 0.7363349921087369


# 2.2

## Linear Regression

In [31]:
from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.fit(x_train_sc, y_train_)
ypr3 = model.predict(x_test_sc)
print(f'RMSE: {root_mean_squared_error(y_test_, ypr3)}')

RMSE: 0.7455813830127761


## Tensorflow

In [36]:
model_nn = tf.keras.Sequential([
    tf.keras.layers.Dense(10, activation='tanh'),
    tf.keras.layers.Dense(1)
])
model_nn.compile(optimizer = tf.keras.optimizers.SGD(learning_rate=0.001),
                loss = 'mse'
                )
model_nn.fit(x_train_sc, y_train_sc, epochs=50, batch_size=32)

ypr4 = sc_y.inverse_transform(model_nn.predict(x_test_sc))
print(f'RMSE: {root_mean_squared_error(y_test_, ypr4)}')

Epoch 1/50
516/516 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 1.1634
Epoch 2/50
516/516 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 1.0879
Epoch 3/50
516/516 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 1.0376
Epoch 4/50
516/516 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.9930
Epoch 5/50
516/516 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.9519
Epoch 6/50
516/516 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.9126
Epoch 7/50
516/516 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.8746
Epoch 8/50
516/516 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.8365
Epoch 9/50
516/516 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.7983
Epoch 10/50
516/516 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.7599
Epoch 11/50
516/516 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.7218
Epoch 12/50
516/516 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.6839
Epoch 13/50
516/516 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.6472
Epoch 14/50
516/516 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.6121
Epoch 15/50
516/516 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - lo